# Notebook 01 — Fundamentos da API do Gemini

Este notebook cobre a API do Gemini do zero, usando o SDK `google-genai` (versao nova).
Ao terminar, voce vai entender exatamente o que acontece dentro do `agent.py` do curso
quando ele chama o modelo, faz streaming de tokens e executa ferramentas.

**O que vamos ver:**

1. Instalacao e configuracao do SDK
2. Chamada basica de geracao de texto
3. Parametros do modelo (temperatura, max tokens)
4. Streaming de respostas token a token
5. Tool Calling (Function Calling) — o mecanismo central dos agentes
6. Exercicios praticos
7. Conexao com o codigo do curso

In [ ]:
# Instala as dependencias necessarias para este notebook
!uv pip install google-genai rich

import os
from google import genai
from google.genai import types
from rich.console import Console
from rich.panel import Panel
from rich.syntax import Syntax

console = Console()

# Carrega a chave da API a partir da variavel de ambiente
api_key = os.environ.get("GEMINI_API_KEY")
if not api_key:
    raise EnvironmentError(
        "Variavel GEMINI_API_KEY nao encontrada.\n"
        "Execute: export GEMINI_API_KEY='sua-chave-aqui'"
    )

# Cria o cliente — ponto de entrada para todas as chamadas
client = genai.Client(api_key=api_key)

console.print("[bold green]SDK configurado com sucesso.[/]")

## 1. Chamada basica

A forma mais simples de usar o Gemini e chamar `client.models.generate_content()`.
Voce passa o nome do modelo e o conteudo (texto, imagem, etc.).
O modelo retorna um objeto `GenerateContentResponse` com o texto gerado em `response.text`.

In [ ]:
# Chamada basica: pergunta simples ao Gemini 2.5 Flash
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Em uma frase, o que e um agente de IA?",
)

# Exibe a resposta com formatacao Rich
console.print(Panel(
    response.text,
    title="Resposta do Gemini",
    border_style="cyan",
))

# O objeto de resposta contem metadados uteis
console.print(f"\n[dim]Tokens de entrada: {response.usage_metadata.prompt_token_count}[/]")
console.print(f"[dim]Tokens de saida:   {response.usage_metadata.candidates_token_count}[/]")

## 2. Parametros do modelo

O comportamento do modelo pode ser controlado com `GenerateContentConfig`:

- **`temperature`** — controla a aleatoriedade da geracao. `0.0` = deterministico (sempre a mesma resposta para o mesmo prompt). `1.0` = mais criativo e variado.
- **`max_output_tokens`** — limite de tokens na resposta. Util para controlar custo e latencia.
- **`top_p` / `top_k`** — parametros alternativos de amostragem (menos usados no dia a dia).

A celula abaixo chama o modelo duas vezes com o mesmo prompt mas temperaturas diferentes,
para que voce veja a diferenca na pratica.

In [ ]:
prompt = "Crie um nome criativo para uma startup de tecnologia financeira."

for temperature in [0.0, 1.0]:
    config = types.GenerateContentConfig(
        temperature=temperature,
        max_output_tokens=100,
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=config,
    )

    console.print(Panel(
        response.text.strip(),
        title=f"temperature={temperature}",
        border_style="yellow" if temperature == 0.0 else "magenta",
    ))

console.print("\n[dim]Execute a celula varias vezes para ver o temperature=1.0 variar.[/]")

## 3. Streaming de respostas

Por padrao, `generate_content()` aguarda a resposta completa antes de retornar.
Para experiencias responsivas (como um chat que exibe palavras conforme chegam),
use `generate_content_stream()`, que retorna um iterador de chunks.

Cada chunk tem `.text` com o fragmento de texto gerado naquele momento.
Esse e exatamente o mecanismo que o Stage 02 do curso expoe via SSE.

In [ ]:
import sys

prompt = "Explique em tres paragrafos curtos como funciona o protocolo HTTP."

console.print("[bold cyan]Streaming iniciado:[/]\n")

# generate_content_stream retorna um iterador
# Cada chunk chegou do servidor conforme foi gerado
chunk_count = 0
for chunk in client.models.generate_content_stream(
    model="gemini-2.5-flash",
    contents=prompt,
):
    # Exibe o fragmento sem quebra de linha (flush imediato)
    if chunk.text:
        print(chunk.text, end="", flush=True)
        chunk_count += 1

print()  # Quebra de linha final
console.print(f"\n[dim]Total de chunks recebidos: {chunk_count}[/]")

## 4. Tool Calling (Function Calling)

Tool Calling e o mecanismo que transforma um modelo de linguagem em um agente.
Em vez de responder diretamente, o modelo pode decidir chamar uma funcao que voce definiu.

O fluxo e:

1. Voce define ferramentas como funcoes Python com docstring
2. Voce envia uma mensagem e a lista de ferramentas
3. O modelo responde com um `tool_call` (nome + argumentos) em vez de texto
4. Voce executa a funcao localmente com os argumentos
5. Voce devolve o resultado ao modelo
6. O modelo gera a resposta final em linguagem natural

No `agent.py` do curso, o LangGraph automatiza os passos 3-6 em loop.

In [ ]:
import json

# --- Passo 1: Definir a ferramenta como funcao Python ---

def calcular_media(numeros: list[float]) -> float:
    """Calcula a media aritmetica de uma lista de numeros.

    Args:
        numeros: Lista de valores numericos.

    Returns:
        Media aritmetica dos valores.
    """
    if not numeros:
        return 0.0
    return sum(numeros) / len(numeros)


# --- Passo 2: Enviar mensagem com a ferramenta disponivel ---

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Qual a media dos valores: 450000, 620000, 390000 e 510000?",
    config=types.GenerateContentConfig(
        tools=[calcular_media],  # SDK converte a funcao automaticamente
        temperature=0,
    ),
)

# --- Passo 3: Inspecionar o tool_call retornado ---

parte = response.candidates[0].content.parts[0]
tool_call = parte.function_call

console.print(Panel(
    f"Ferramenta: [bold]{tool_call.name}[/]\n"
    f"Argumentos: {dict(tool_call.args)}",
    title="Modelo solicitou uma ferramenta",
    border_style="yellow",
))

# --- Passo 4: Executar a funcao localmente ---

args = dict(tool_call.args)
resultado = calcular_media(**args)
console.print(f"\n[dim]Resultado local: {resultado}[/]")

# --- Passo 5 e 6: Enviar resultado de volta e obter resposta final ---

response_final = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        types.Content(
            role="user",
            parts=[types.Part(text="Qual a media dos valores: 450000, 620000, 390000 e 510000?")],
        ),
        # Resposta do modelo com o tool_call
        response.candidates[0].content,
        # Resultado da execucao local da ferramenta
        types.Content(
            role="user",
            parts=[
                types.Part(
                    function_response=types.FunctionResponse(
                        name=tool_call.name,
                        response={"resultado": resultado},
                    )
                )
            ],
        ),
    ],
    config=types.GenerateContentConfig(
        tools=[calcular_media],
        temperature=0,
    ),
)

console.print(Panel(
    response_final.text,
    title="Resposta final do modelo",
    border_style="green",
))

## Tente voce

Tres exercicios para praticar o que foi visto:

**Exercicio 1 — Multiturno:**
Adapte a celula de chamada basica para fazer uma conversa de dois turnos.
Envie uma pergunta, receba a resposta, depois envie uma pergunta de acompanhamento
usando o historico completo em `contents` (lista de `types.Content`).

**Exercicio 2 — Streaming com contagem:**
Modifique a celula de streaming para contar quantas palavras chegaram em cada chunk.
Exiba a contagem ao lado de cada fragmento.

**Exercicio 3 — Segunda ferramenta:**
Defina uma segunda funcao Python (por exemplo, `converter_moeda(valor: float, taxa: float) -> float`)
e envie uma mensagem que force o modelo a escolher entre as duas ferramentas.

## Conexao com o curso

O `agent.py` usado em todos os stages do curso usa `langchain-google-genai` e `langgraph`,
que sao camadas de abstracao sobre exatamente o que foi visto neste notebook:

| Conceito visto aqui | Como aparece no agent.py |
|---|---|
| `client.models.generate_content()` | `ChatGoogleGenerativeAI(model="gemini-2.5-flash")` |
| `GenerateContentConfig(temperature=0)` | `ChatGoogleGenerativeAI(..., temperature=0)` |
| `generate_content_stream()` | `agent.astream(state, stream_mode="updates")` |
| Tool Calling manual (passos 1-6) | `create_react_agent(model, tools=[...])` — o LangGraph fecha o loop automaticamente |
| `tool_call.name` / `tool_call.args` | `AgentEvent(type="tool_call", data={...})` |

A diferenca principal e que o LangGraph repete o ciclo tool_call → execucao → resultado
quantas vezes for necessario, ate o modelo decidir que tem informacao suficiente para responder.
Esse loop e o que transforma uma chamada de API em um agente.